In [ ]:
# ------------------------------------------------------------------
# Step 1 — Build RQ1 BLS → O*NET occupation universe
# Evidence class: Data preparation
# ------------------------------------------------------------------

from pathlib import Path
import zipfile, io
import pandas as pd

ROOT = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes/01_BLS_Raw_Data_Construct")

BLS_ZIP = ROOT / "Input/oesm25in4.zip"
ONET_FILE = ROOT / "Input/Occupation Data.xlsx"
TASK_FILE = ROOT / "Input/Task Statements.xlsx"

def read_bls_file(token):
    with zipfile.ZipFile(BLS_ZIP) as z:
        name = next(
            n for n in z.namelist()
            if token in n.lower() and n.lower().endswith(".xlsx")
        )
        return pd.read_excel(io.BytesIO(z.read(name)), dtype=str)

# BLS industry occupation files
sector = read_bls_file("natsector")
industry4 = read_bls_file("nat4d")

samples = []

for name, naics in {
    "Finance": "52",
    "Education": "61",
    "Healthcare": "62",
}.items():

    x = sector[
        (sector["NAICS"].str.strip() == naics) &
        (sector["O_GROUP"].str.lower() == "detailed")
    ][["OCC_CODE", "OCC_TITLE"]].copy()

    x["sector"] = name
    samples.append(x)

# Legal Services
x = industry4[
    (industry4["NAICS"].str.replace("-", "").str.strip() == "541100") &
    (industry4["O_GROUP"].str.lower() == "detailed")
][["OCC_CODE", "OCC_TITLE"]].copy()

x["sector"] = "Legal"
samples.append(x)

bls_occ = pd.concat(samples, ignore_index=True)
bls_occ["soc6"] = bls_occ["OCC_CODE"].str[:7]

# O*NET 30.0
onet = pd.read_excel(ONET_FILE, dtype=str)

onet["soc6"] = onet["O*NET-SOC Code"].str[:7]

study_onet = bls_occ.merge(
    onet[["O*NET-SOC Code", "Title", "soc6"]],
    on="soc6",
    how="inner"
)

study_onet = study_onet.rename(columns={
    "OCC_TITLE": "occupation_title",
    "O*NET-SOC Code": "onet_soc_code",
    "Title": "onet_title"
})

study_onet = study_onet[
    ["sector", "soc6", "occupation_title",
     "onet_soc_code", "onet_title"]
].drop_duplicates()


# Preserve sector membership at the SOC6 level
sector_membership = (
    pd.crosstab(
        study_onet["soc6"],
        study_onet["sector"]
    )
    .gt(0)
    .reset_index()
    .rename(columns={
        "Finance": "sector_finance",
        "Education": "sector_education",
        "Healthcare": "sector_healthcare",
        "Legal": "sector_legal",
    })
)


In [ ]:
# Save Step 1 output
OUT = ROOT / "Output"

initial_onet = (
    study_onet
    .drop_duplicates("onet_soc_code")
    .sort_values(["soc6", "onet_soc_code"])
    .reset_index(drop=True)
)

initial_onet.to_csv(
    OUT / "ONET_Occupation_Universe_Initial.csv",
    index=False
)

print("Unique O*NET-SOC:", initial_onet["onet_soc_code"].nunique())
print("Unique SOC6:", initial_onet["soc6"].nunique())

In [ ]:
# ------------------------------------------------------------------
# Step 2 — Harmonize to parent SOC6 universe
# Evidence class: Data preparation
# ------------------------------------------------------------------

parent_644 = (
    study_onet[
        ["soc6", "occupation_title", "onet_soc_code", "onet_title"]
    ]
    .assign(is_00=lambda x: x["onet_soc_code"].str.endswith(".00"))
    .sort_values(["soc6", "is_00"], ascending=[True, False])
    .drop_duplicates("soc6")
    .drop(columns="is_00")
    .reset_index(drop=True)
)

print("Parent SOC6:", parent_644["soc6"].nunique())

Which of the 644 SOC6 occupations have no task statements anywhere under that SOC6 in O*NET?

In [ ]:
# Step 3 — Identify unrecoverable parent SOC6
# Evidence class: Data preparation
# ------------------------------------------------------------------
tasks = pd.read_excel(TASK_FILE, dtype=str)
tasks["soc6"] = tasks["O*NET-SOC Code"].str[:7]

unrecoverable = parent_644[
    ~parent_644["soc6"].isin(tasks["soc6"])
].copy()

print("Unrecoverable SOC6:", unrecoverable["soc6"].nunique())

Which parent SOC6 occupations lack tasks for the retained ONET code but have task statements available under detailed child ONET codes?

In [ ]:
# ------------------------------------------------------------------
# Step 4 — Identify recoverable parent SOC6
# Evidence class: Data preparation
# ------------------------------------------------------------------

recoverable = parent_644[
    ~parent_644["onet_soc_code"].isin(tasks["O*NET-SOC Code"]) &
    parent_644["soc6"].isin(tasks["soc6"])
].copy()

print("Recoverable SOC6:", recoverable["soc6"].nunique())

Which parent SOC6 occupations have task statements directly available for the retained O*NET occupation code?

In [ ]:
# ------------------------------------------------------------------
# Step 5 — Identify direct task matches
# Evidence class: Data preparation
# ------------------------------------------------------------------

direct_matches = parent_644[
    parent_644["onet_soc_code"].isin(tasks["O*NET-SOC Code"])
].copy()

print("Direct task matches:", direct_matches["soc6"].nunique())

Which SOC6 occupations remain analyzable after combining direct task matches with recoverable parents?

In [ ]:
# ------------------------------------------------------------------
# Step 6 — Build final SOC6 analytic universe
# Evidence class: Data preparation
# ------------------------------------------------------------------

final_soc6 = pd.concat(
    [direct_matches, recoverable],
    ignore_index=True
).drop_duplicates("soc6")

print("Final SOC6 analytic universe:", final_soc6["soc6"].nunique())

Which O*NET occupations actually supply task statements for the final 604 SOC6 occupations?

In [ ]:
# ------------------------------------------------------------------
# Step 7 — Build O*NET task-supplier universe
# Evidence class: Data preparation
# ------------------------------------------------------------------

# Direct task suppliers
direct_suppliers = direct_matches[
    ["soc6", "onet_soc_code", "occupation_title"]
].copy()

direct_suppliers["task_source_type"] = "direct"

# Child O*NET occupations supplying tasks for recoverable parents
child_suppliers = (
    tasks[tasks["soc6"].isin(recoverable["soc6"])]
    [["soc6", "O*NET-SOC Code"]]
    .drop_duplicates()
    .rename(columns={"O*NET-SOC Code": "onet_soc_code"})
)

child_suppliers = child_suppliers.merge(
    parent_644[["soc6", "occupation_title"]],
    on="soc6",
    how="left"
)

child_suppliers["task_source_type"] = "recovered_child"

# Final supplier universe
task_suppliers = pd.concat(
    [direct_suppliers, child_suppliers],
    ignore_index=True
)

print("SOC6:", task_suppliers["soc6"].nunique())
print("O*NET occupations supplying tasks:", task_suppliers["onet_soc_code"].nunique())

In [ ]:
# Standardize O*NET task fields
tasks = tasks.rename(columns={
    "O*NET-SOC Code": "onet_soc_code",
    "Task ID": "task_id",
    "Task": "task_statement",
    "Task Type": "task_type"
})

In [ ]:
task_universe = (
    task_suppliers.merge(
        tasks[
    [
    "onet_soc_code",
    "task_id",
    "task_statement",
    "task_type",
    "Incumbents Responding",
    "Date",
    "Domain Source",
    ]
        ],
        on="onet_soc_code",
        how="inner"
    )
    .drop_duplicates(["onet_soc_code", "task_id"])
)

print("Occupation-task records:", len(task_universe))

In [ ]:
# Keep respondent counts as nullable integers
task_universe["Incumbents Responding"] = (
    task_universe["Incumbents Responding"].astype("Int64")
)

In [ ]:
task_universe = task_universe.merge(
    sector_membership,
    on="soc6",
    how="left",
    validate="many_to_one"
)

In [ ]:
# Save final task universe
task_universe.to_csv(
    ROOT / "Output/ONET_Task_Universe.csv",
    index=False
)